## **Uma análise de técnicas de PLN para detecção de Fake News no contexto da Álgebra Linear**
### SME0142 - Álgebra Linear e Aplicações
---
* **Docente:** Cynthia de Oliveira Lage Ferreira
* **Alunos:** Pedro Henrique de Sousa Prestes (15507819)
---

### Introdução
Esse trabalho consiste em discutir, na perspectiva da **Álgebra Linear**, as técnicas de Processamento de Linguagem Natural (PLN) utilizadas no artigo *Contributions to the Study of Fake News in Portuguese: New Corpus and Automatic Detection Results* publicado por pesquisadores do Núcleo Interinstitucional de Linguística Computacional do ICMC/USP e da UFSCar. Assim podendo avaliar o impacto desse campo da matemática para um problema muito pertinente nos dias de hoje e de que forma a metodologia de estudos em linguagem natural é modelada pensando em elementos como vetores e transformações. Não é necessário que a pessoa tenha lido o artigo previamente.

In [1]:
# Setup
%pip install scikit-learn matplotlib pandas seaborn nltk --quiet

import os
import re
import pandas as pd
import numpy as np
import subprocess
import nltk
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import cross_validate
from sklearn.decomposition import TruncatedSVD
from nltk.corpus import stopwords
from nltk.stem import RSLPStemmer

Note: you may need to restart the kernel to use updated packages.


---

### Fake.br-Corpus
Uma preocupação válida do artigo em questão é sobre a disponibilidade de um bom dataset de fake news brasileiras. Pensando nisso, é proposto o Fake.br-Corpus, que traz uma série de notícias de diferentes fontes com sua veracidade analisada manualmente, contendo **3600 notícias falsas e 3600 notícias verdadeiras**. O dataset está público em um repositório do Github e será usado para simular a modelagem do modelo de aprendizado de máquina proposto no artigo.

In [2]:
if not os.path.exists('Fake.br-Corpus'):
    try:
        print('Clonando o repositório do Fake.br-Corpus...')
        subprocess.run(['git', 'clone', 'https://github.com/roneysco/Fake.br-Corpus.git'])
    except subprocess.CalledProcessError as e:
        print(f'Falha em clonar o repositório do Fake.br-Corpus: {e}')

Clonando o repositório do Fake.br-Corpus...


Cloning into 'Fake.br-Corpus'...
Updating files: 100% (21602/21602), done.


---

### Bag of Words
O método Bag of Words (Saco de Palavras) é a **técnica mais fundamental em PLN para transformar texto em vetores numéricos**, algo que modelos de aprendizado de máquina sabem trabalhar. Podemos interpretá-lo da seguinte forma:
- Imagine que $V$ é o conjunto de todas as palavras únicas que existem no seu corpus (o vocabulário). Se você tem 10.000 palavras únicas, seu espaço vetorial é $\mathbb{R}^{10.000}$

- Cada documento (notícia) é um vetor $d$, com a coordenada $d_i$ representando a frequência daquela palavra no documento ou somente se ela está presente ou não (binário). Apesar de o artigo dizer que o uso de frequências não melhorou em relação ao método binário, utilizaremos as frequências em um primeiro momento, pois é o padrão da indústria para identificar o grau da importância de palavras, muito usado em LLMs também.

- O nome "Saco de Palavras" vem da ideia de que a estrutura da frase é perdida. "O cão mordeu o homem" e "O homem mordeu o cão" geram exatamente o mesmo vetor, pois contêm as mesmas palavras.

In [3]:
# Corpus: Amostras retiradas da Tabela 1 do artigo (Fake vs True).
corpus = [
    # Texto 0 (Fake)
    """Michel Temer propõe fim do carnaval por 20 anos, “PEC dos gastos”. Michel Temer 
    afirmou que não deve haver gastos com aparatos supérfluos sem pensar primeiraente 
    na educação do Brasil. A medida pretende calcelar o carnaval de 2018.",
    """,
    # Texto 1 (True - Alinhada à anterior)
    """Michel Temer não quer o fim do Carnaval por 20 anos. Notícias falsas misturam
    proximidade dos festejos, crise econômica e medidas impopulares do governo do
    peemedebista.
    """,
    # Texto 2 (Fake - Outro assunto)
    """Acabou a mordomia ! Ingresso mais barato pra mulher é ilegal. Baladas que davam
    meia entrada para mulher, ou até mesmo gratuidade, esto na ilegalidade agora.
    Acabou o preconceito com os homens nas casas de show de todo o Brasil.
    """,
    # Texto 3 (True - Alinhada à anterior)
    """Ingresso feminino barato como marketing 'não inferioriza mulher', diz juíza do DF.
    Afirmação consta em decisão sobre preços diferentes para homens e mulheres em festa
    no Lago Paranóia. 'Prática permite que mulher possa optar por participar de tais
    eventos sociais', diz texto.
    """
]

labels = ['Fake_Carnaval', 'True_Carnaval', 'Fake_Ingresso', 'True_Ingresso']

# O artigo menciona remoção de stopwords (palavras desnecessárias), estamos simulando a remoção de artigos e conectivos.
stop_words = ['o', 'a', 'do', 'da', 'de', 'em', 'para', 'por', 'e', 'que', 'é']

# Instanciando o vetorizador (transforma texto em R^N)
vectorizer_bow = CountVectorizer(stop_words=stop_words)

# X_bow é a nossa Matriz de Termos-Documentos
X_bow = vectorizer_bow.fit_transform(corpus)

# Convertendo para DataFrame para facilitar a visualização algébrica
df_bow = pd.DataFrame(X_bow.T.toarray(), columns=labels, index=vectorizer_bow.get_feature_names_out())

# Visualizando apenas algumas colunas (coordenadas) interessantes
coords_interesse = ['carnaval', 'temer', 'ilegal', 'mulher']
print(f"\nDimensão do Espaço Vetorial (Vocabulário): R^{df_bow.shape[0]}")
print(f"Quantidade de vetores (Notícias) no Espaço Vetorial: {df_bow.shape[1]}")
df_bow.loc[coords_interesse]


Dimensão do Espaço Vetorial (Vocabulário): R^92
Quantidade de vetores (Notícias) no Espaço Vetorial: 4


,Fake_Carnaval,True_Carnaval,Fake_Ingresso,True_Ingresso
carnaval,2,1,0,0
temer,2,1,0,0
ilegal,0,0,1,0
mulher,0,0,2,2


---
### TF-IDF: Reescalonamento dos Eixos
O **TF-IDF (Term Frequency - Inverse Document Frequency)** é uma transformação aplicada sobre a matriz BoW para corrigir distorções de escala. Perceba que a matriz que criamos simplesmente conta a quantidade de vezes que cada palavra aparece nas notícias, porém o **TF-IDF é o real padrão da indústria** para identificação de frequências de forma a "dar mais peso" ao que importa.

#### A Matemática do TF-IDF
Na perspectiva de uma transformação na Álgebra Linear: **é uma multiplicação de matrizes elemento-a-elemento que "estica" ou "encolhe" os eixos do espaço vetorial**:
- TF (Term Frequency): Mede a "força" da palavra no documento local.
- IDF (Inverse Document Frequency): Mede a "raridade" da palavra no universo global. Se a palavra "que" aparece em 100% dos documentos, ela não serve para separar notícias Falsas de Verdadeiras. O IDF joga o valor dessa coordenada para perto de zero.

A fórmula para o peso $w$ de um termo $t$ em um documento $d$:
$$
w_{t,d} = tf_{t,d} ​× log(\frac{N}{df_t}​​)
$$

Onde:
- **$tf$ (Term Frequency)**: Representa a frequência do termo no documento, varia de acordo com a implmentação do TF-IDF.
- **$N$**: É a quantidade de documentos.
- **$df$ (Document Frequency)**: É a frequência de um termo entre todos os documentos.

#### TF-IDF como Transformação Linear
Porém, na perspectiva da Álgebra Linear, o TF-IDF pode ser interpretado não como uma simples multiplicação de escalares, mas como uma **transformação linear que atua no espaço vetorial** que nós definimos entre palavras e documentos. 

Se considerarmos nossa matriz de frequências T (onde cada coluna é um documento e cada linha é um termo), a **aplicação dos pesos IDF equivale a multiplicar a matriz diagonal I por uma matriz T**.

Sendo I a matriz diagonal dos pesos e T a matriz de TFs:

$$I = \begin{pmatrix} log(\frac{N}{df_1}​​) & 0 & 0 \\ 0 & log(\frac{N}{df_2}​​) & 0 \\ 0 & 0 & log(\frac{N}{df_3}​​) \end{pmatrix}$$

$$T = \begin{pmatrix} tf_{1,1} & tf_{1,2} & tf_{1,3} \\ tf_{2,1} & tf_{2,2} & tf_{2,3} \\ tf_{3,1} & tf_{3,2} & tf_{3,3} \end{pmatrix}$$

Temos que:

$$W = I ​× T \Longrightarrow w_{t,d} = tf_{t,d} ​× log(\frac{N}{df_t}​​)$$

Ao analisarmos a matriz de transformação IDF sob a ótica espectral, observamos que ela já se encontra diagonalizada na base canônica do espaço de palavras. Isso implica que **cada palavra do vocabulário é um autovetor dessa transformação, e o seu autovalor correspondente é o seu peso IDF**.

Geometricamente, isso confirma que **a transformação IDF atua puramente como um operador de escala (scaling), esticando ou encolhendo os eixos do espaço sem rotacioná-los.** Ela preserva a independência semântica das palavras (ortogonalidade), **alterando apenas a magnitude (importância) de cada dimensão para a classificação.**

In [4]:
# O TF-IDF penaliza palavras que aparecem em todos os documentos e valoriza as raras
vectorizer_tfidf = TfidfVectorizer(stop_words=stop_words)
X_tfidf = vectorizer_tfidf.fit_transform(corpus)

df_tfidf = pd.DataFrame(X_tfidf.T.toarray(), columns=labels, index=vectorizer_tfidf.get_feature_names_out())
df_tfidf.loc[coords_interesse].round(4)

,Fake_Carnaval,True_Carnaval,Fake_Ingresso,True_Ingresso
carnaval,0.2822,0.1926,0.0000,0.0000
temer,0.2822,0.1926,0.0000,0.0000
ilegal,0.0000,0.0000,0.1790,0.0000
mulher,0.0000,0.0000,0.2822,0.2675


---

Ok, até agora estamos vendo a parte de experimentação do artigo, ou seja, como eles utilizaram o dataset que eles haviam criado para definir métodos de aprendizado de máquina e atingir os melhores resultados. Entretanto, vamos voltar para o processo de criação do dataset.

Dado as notícias coletadas, os autores tiveram que encontrar uma maneira de criar pares de notícia do tipo (verdadeiro, falso), assim como os 4 que estivemos usando de exemplo até agora. Uma forma possível seria fazer isso manualmente, pesquisar uma notícia verdadeira e achar o seu correspondente falso e vice-versa, porém o que foi feito é algo mais inteligente e que também usa os conceitos de Álgebra Linear! Já que agora estamos munidos de conhecimento de como podemos representar os documentos em um espaço vetorial de coordenadas com frequências de palavras (tanto no método de contagem quanto com TF-IDF), nós podemos entender de que forma isso é feito.

---
### Medida de similaridade lexical (cosseno)
A similaridade entre duas palavras pode ser descrita como a **similaridade entre os seus dois vetores numéricos correspondentes**. Notícias muito similares provavelmente discorrem sobre os mesmos assuntos. A maneira como essa similaridade é calculada é por meio do **ângulo entre os vetores**:
$$sim(A,B) = \cos(\theta) = \frac{A \cdot B}{||A||||B||}$$
Ou seja, estamos lidando com um **espaço vetorial euclidiano** que admite o produto escalar convencional. Isso é muito importante para definirmos **ortogonalidade**, afinal, notícias com similaridade 0 são vetores ortogonais dentro desse espaço euclidiano e sem nenhuma relação.

In [5]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Indice do vetor que estaremos analisando
v = 0

print(f"=== Medidade de similaridade com: '{labels[v]}' ===\n")

# 1. Pegamos o vetor da Fake News de Carnaval
vetor_fake_carnaval = X_tfidf[v]

# 2. Calculamos o cosseno contra os documentos da matriz
# Isso retorna uma lista de scores (cos(theta))
similaridades = cosine_similarity(vetor_fake_carnaval, X_tfidf)[0]
indices_ordenados = np.argsort(similaridades)[::-1] # [::-1] inverte para decrescente

# O primeiro da lista sempre será ele mesmo (Score 1.0, Ângulo 0), então pulamos o índice 0
melhor_match_idx = indices_ordenados[1] 

for i in indices_ordenados:
    if i == v: continue
    
    score = similaridades[i]

    # Converter Similaridade (Cos) -> Ângulo (Graus)
    # theta = arccos(produto_interno)
    # Clip em 1.0 para evitar erros numéricos de ponto flutuante (ex: 1.00000001)
    theta_rad = np.arccos(min(score, 1.0)) 
    theta_graus = np.degrees(theta_rad)
    
    print(f"Comparando com: {labels[i]}")
    print(f"  > Similaridade (Cos): {score:.4f}")
    print(f"  > Ângulo Geométrico:  {theta_graus:.2f}°")
    print("-" * 30)

print(f"A notícia '{labels[v]}' foi melhor alinhada com '{labels[melhor_match_idx]}'.")

=== Medidade de similaridade com: 'Fake_Carnaval' ===

Comparando com: True_Carnaval
  > Similaridade (Cos): 0.2896
  > Ângulo Geométrico:  73.17°
------------------------------
Comparando com: Fake_Ingresso
  > Similaridade (Cos): 0.0597
  > Ângulo Geométrico:  86.58°
------------------------------
Comparando com: True_Ingresso
  > Similaridade (Cos): 0.0124
  > Ângulo Geométrico:  89.29°
------------------------------
A notícia 'Fake_Carnaval' foi melhor alinhada com 'True_Carnaval'.


---

Agora que aprendemos um pouco sobre como o dataset foi feito, podemos voltar para as técnicas de modelos em Machine Learning.

### O Classificador: Support Vector Machine (SVM) Linear

O **SVM (Máquina de Vetores de Suporte)** é um algoritmo de aprendizado supervisionado que, no contexto deste trabalho, atua como um **Classificador Linear**.

Do ponto de vista da Álgebra Linear, o objetivo do SVM é encontrar um **Hiperplano Separador Ótimo** que divida o espaço vetorial $\mathbb{R}^N$ (nosso vocabulário) em duas regiões distintas: uma contendo as notícias falsas e outra contendo as verdadeiras.

#### A Matemática do Hiperplano

Um hiperplano em um espaço de $N$ dimensões é definido pela equação linear:

$$w \cdot x + b = 0$$

Onde:
- **$x$**: É o vetor de entrada (a notícia vetorizada pelo TF-IDF).
- **$w$ (Weight Vector)**: É um vetor normal ao hiperplano. Ele define a orientação da fronteira de decisão. Cada coordenada de $w$ representa o peso (importância) de uma palavra para a decisão.
- **$b$ (Bias)**: É um escalar que representa o deslocamento do hiperplano em relação à origem.

#### O Critério de Decisão (Projeção Ortogonal)

Para classificar uma nova notícia $x_{nova}$, o modelo calcula a projeção desse ponto na direção normal ao plano:

$$f(x) = w \cdot x_{nova} + b$$

A classificação é determinada puramente pelo **sinal** dessa operação de produto interno:
- Se $f(x) > 0$: A notícia está no lado "Positivo" do hiperplano (Classe A).
- Se $f(x) < 0$: A notícia está no lado "Negativo" do hiperplano (Classe B).
- Se $f(x) = 0$: A notícia está exatamente sobre a fronteira (indecisão).

#### Por que "Vetores de Suporte"?

Dentre infinitos planos possíveis que separam os dados, o SVM escolhe aquele que maximiza a **Margem**, a distância entre o hiperplano e os pontos de dados mais próximos de cada classe. Esses pontos cruciais que "seguram" ou "suportam" a fronteira são chamados de **Vetores de Suporte**.

#### Implementação Específica: LinearSVC

O artigo base utiliza especificamente a implementação **LinearSVC** (*Support Vector Classification with Linear Kernel*) do Scikit-Learn.

Do ponto de vista computacional e de Álgebra Linear, esta escolha é estratégica por dois motivos:

1.  Alta Dimensionalidade: Como vetores de texto geram um espaço de características muito vasto, a separação entre classes geralmente pode ser feita de forma linear. Isso dispensa o uso do "Kernel Trick" (técnica para projetar dados em dimensões ainda maiores), pois a própria complexidade natural do texto já permite traçar uma fronteira de decisão clara.

2. Eficiência no Problema Primal: Ao contrário do SVM tradicional, o LinearSVC resolve diretamente o problema primal de otimização. Isso o torna extremamente rápido para matrizes esparsas (como o TF-IDF), pois o algoritmo foca diretamente nos coeficientes da reta, evitando a sobrecarga matemática de calcular kernels complexos para cada par de dados.

In [6]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

# 0 = Fake, 1 = True (Vamos convencionar assim para o SVM)
# O SVM vai tentar separar essas duas classes.
y = np.array([0, 1, 0, 1]) 

# Utilizamos a vetorização por TF-IDF que fizemos anteriormente
X = X_tfidf

# 3. Treinamento do SVM (Encontrando o Hiperplano)
# O artigo cita o uso do LinearSVC 
svm = LinearSVC(random_state=42)
svm.fit(X, y)

# O atributo .coef_ retorna o vetor Normal (w) ao hiperplano
# O atributo .intercept_ retorna o viés (b), que é a distância da origem
w = svm.coef_[0]   # Vetor w
b = svm.intercept_[0] # Escalar b

print("=== EQUAÇÃO DO HIPERPLANO ===")
print(f"Dimensão do Espaço (Vocabulário): R^{len(w)}")
print(f"Viés (b): {b:.4f}")
print("Vetor Normal (w) - mostrando as 5 dimensões (palavras) de maior peso:")

# Vamos ver quais palavras 'empurram' o vetor para o lado Fake ou True
vocab = vectorizer_tfidf.get_feature_names_out()
indices_ordenados = np.argsort(w) # Ordena do menor (mais negativo) para o maior (mais positivo)

# Top palavras que indicam CLASSE 0 (Fake - Lado Negativo do Hiperplano neste treino)
print("\nPalavras que puxam para o lado FAKE (Pesos Negativos):")
for i in indices_ordenados[:3]:
    print(f"  Word: '{vocab[i]}' | Coef (wi): {w[i]:.4f}")

# Top palavras que indicam CLASSE 1 (True - Lado Positivo do Hiperplano)
print("\nPalavras que puxam para o lado TRUE (Pesos Positivos):")
for i in indices_ordenados[::-1][:3]:
    print(f"  Word: '{vocab[i]}' | Coef (wi): {w[i]:.4f}")

# Vamos pegar a alguma notícia e projetar no vetor w e comparar com o resultado que conhecemos
print("\n=== CLASSIFICAÇÃO MANUAL ===")
v = 0
noticia_teste = X[v].toarray()[0] # Vetor x

# A Fórmula fundamental: f(x) = w . x + b
# Produto Interno (Dot Product)
projecao = np.dot(w, noticia_teste) + b

print(f"Texto: '{corpus[v][:60]}...'")
print(f"Resultado Algébrico (w . x + b): {projecao:.4f}")

# Decisão baseada no sinal
classe_predita = 1 if projecao > 0 else 0
classe_real = y[v]

print(f"Classificação: {'VERDADEIRO' if classe_predita == 1 else 'FAKE'}")
print(f"O modelo acertou? {classe_predita == classe_real}")

=== EQUAÇÃO DO HIPERPLANO ===
Dimensão do Espaço (Vocabulário): R^92
Viés (b): 0.0117
Vetor Normal (w) - mostrando as 5 dimensões (palavras) de maior peso:

Palavras que puxam para o lado FAKE (Pesos Negativos):
  Word: 'gastos' | Coef (wi): -0.2892
  Word: 'acabou' | Coef (wi): -0.2526
  Word: 'com' | Coef (wi): -0.2136

Palavras que puxam para o lado TRUE (Pesos Positivos):
  Word: 'diz' | Coef (wi): 0.2438
  Word: 'quer' | Coef (wi): 0.1971
  Word: 'peemedebista' | Coef (wi): 0.1971

=== CLASSIFICAÇÃO MANUAL ===
Texto: 'Michel Temer propõe fim do carnaval por 20 anos, “PEC dos ga...'
Resultado Algébrico (w . x + b): -0.5959
Classificação: FAKE
O modelo acertou? True


---
### Funcionamento com o Dataset real
Agora que sabemos como muitos dos conceitos por trás da modelagem de ML para esse problema funcionam, podemos finalmente aplicá-los todos juntos em notícias reais e ver o resultado!

Algumas das etapas do pré-processamento haviam sido omitidas para fins didáticos e exemplificação, aqui iremos utilizar todas as técnicas descritas no artigo, como remoção de pontuação e *steeming* a fim de conseguir o resultado mais próximo possível do obtido no artigo.

In [7]:
# Utilizando as notícias de tamanho normalizado para o modelo não ter viés de tamanho das manchetes
dataset_path = 'Fake.br-Corpus/size_normalized_texts' 

def download_nltk_resources():
    """Baixa recursos necessários do NLTK para português."""
    try:
        nltk.data.find('tokenizers/punkt')
        nltk.data.find('corpora/stopwords')
        nltk.data.find('stemmers/rslp')
    except LookupError:
        print("Baixando recursos do NLTK...")
        nltk.download('punkt')
        nltk.download('stopwords')
        nltk.download('rslp')

def load_data(base_path):
    """
    Lê os arquivos da pasta size_normalized_texts.
    O artigo menciona 7.200 notícias (3.600 true, 3.600 fake)
    """
    texts = []
    labels = []
    
    classes = {'fake': 0, 'true': 1} 
    
    print(f"Carregando dados de: {base_path}")
    
    for label_name, label_idx in classes.items():
        folder_path = os.path.join(base_path, label_name)
        
        if not os.path.exists(folder_path):
            print(f"ERRO: A pasta '{folder_path}' não foi encontrada.")
            continue
            
        print(f"Lendo arquivos da pasta: {label_name}...")
        
        # Lista todos os arquivos na pasta
        files = os.listdir(folder_path)

        count = 0
        for filename in files:
            if filename.endswith(".txt"):
                file_path = os.path.join(folder_path, filename)
                try:
                    with open(file_path, 'r', encoding='utf-8') as f:
                        content = f.read()
                        texts.append(content)
                        labels.append(label_idx)
                        count += 1
                except Exception as e:
                    print(f"Erro ao ler {filename}: {e}")
        
        print(f"-> {count} documentos carregados de '{label_name}'.")

    return texts, np.array(labels)

def custom_preprocessor(text):
    """
    Implementa o pré-processamento descrito no artigo:
    1. Case folding (minúsculas)
    2. Remoção de pontuação e números
    3. Stemming (RSLP para português)
    """
    # 1. Case folding
    text = text.lower()
    
    # 2. Remoção de pontuação e números (mantém apenas letras)
    # O artigo remove valores numéricos para evitar esparsidade (nota de rodapé 12) 
    text = re.sub(r'[^a-zmwáàãâéêíóôõúüçñ]+', ' ', text)
    
    tokens = text.split()
    
    # Setup NLTK
    stemmer = RSLPStemmer()
    stops = set(stopwords.words('portuguese'))
    
    # 3. Remoção de stopwords e Stemming
    stems = [stemmer.stem(token) for token in tokens if token not in stops]
    
    return " ".join(stems)

def run_replication():
    download_nltk_resources()
    
    # 1. Carregar Dataset
    texts, labels = load_data(dataset_path)
    
    if len(texts) == 0:
        print("Erro: Nenhum texto encontrado. Verifique o caminho da pasta.")
        return

    print(f"Total de documentos carregados: {len(texts)}")
    
    # 2. Vetorização (Feature Extraction)
    # O artigo usa Bag of Words com valores booleanos (ocorre ou não ocorre)
    print("Vetorizando textos (BoW)...")
    vectorizer = CountVectorizer(
        binary=True,  # O artigo diz que frequência não melhorou, usaram booleanos 
        preprocessor=custom_preprocessor
    )
    
    X = vectorizer.fit_transform(texts)
    y = labels
    
    # 3. Classificação
    # O artigo usa SVM (LinearSVC) com parâmetros default 
    clf = LinearSVC()
    
    # 4. Validação
    # O artigo usa 5-fold cross-validation
    print("Executando 5-fold Cross-Validation...")
    scoring = ['accuracy', 'precision', 'recall', 'f1']
    
    scores = cross_validate(clf, X, y, cv=5, scoring=scoring)
    
    # 5. Exibir Resultados
    print("\n--- Resultados da Replicação (BoW + SVM) ---")
    print(f"Acurácia Média: {scores['test_accuracy'].mean():.4f} (Esperado: ~0.88)")
    print(f"Precisão Média: {scores['test_precision'].mean():.4f}")
    print(f"Recall Médio:   {scores['test_recall'].mean():.4f}")
    print(f"F1-Score Médio: {scores['test_f1'].mean():.4f} (Esperado: ~0.88)")

run_replication()

Carregando dados de: Fake.br-Corpus/size_normalized_texts
Lendo arquivos da pasta: fake...
-> 3600 documentos carregados de 'fake'.
Lendo arquivos da pasta: true...
-> 3600 documentos carregados de 'true'.
Total de documentos carregados: 7200
Vetorizando textos (BoW)...
Executando 5-fold Cross-Validation...

--- Resultados da Replicação (BoW + SVM) ---
Acurácia Média: 0.8826 (Esperado: ~0.88)
Precisão Média: 0.8860
Recall Médio:   0.8783
F1-Score Médio: 0.8821 (Esperado: ~0.88)


---
### Bônus: Análise Semântica Latente (LSA)
Durante o decorrer dessa disciplina, vimos bastante sobre Decomposição em Valores Singulares (SVD) e suas aplicações, dentre elas fizemos um LSA no último trabalho prático. Como estamos falando sobre PLN, nada mais justo que dar um pouco de atenção a esse tópico que já estamos acostumados.

Essa é uma técnica de processamento de linguagem natural que analisa as relações entre um conjunto de documentos e os termos que eles contêm. Ela identifica padrões e conexões entre palavras para entender seu significado contextual e sua similaridade.

#### Categorias das notícias
Na seção do artigo que há a apresentação do Fake.br, é mencionado que elas estão divididas em 6 categorias em que cada notícia foi rotulada manualmente: 
| Category | Number of samples | % |
| :--- | :---: | :---: |
| Politics | 4,180 | 58.0 |
| TV & celebrities | 1,544 | 21.4 |
| Society & daily news | 1,276 | 17.7 |
| Science & technology | 112 | 1.5 |
| Economy | 44 | 0.7 |
| Religion | 44 | 0.7 |

Entretanto, será que esse trabalho poderia ter sido reduzido com o uso de LSA? Será que o SVD consegue diferenciar esses 6 tópicos com número de componentes igual a 6? Vamos tentar descobrir!

In [8]:
texts = []
base_path = 'Fake.br-Corpus/size_normalized_texts'

# Carrega tanto fake quanto true para ver os tópicos globais
for label in ['fake', 'true']:
    folder = os.path.join(base_path, label)
    if os.path.exists(folder):
        for filename in os.listdir(folder):
            if filename.endswith(".txt"):
                with open(os.path.join(folder, filename), 'r', encoding='utf-8') as f:
                    texts.append(f.read())

print(f"{len(texts)} documentos carregados.")

# Pré-processamento simples (só para limpar sujeira)
def simple_clean(text):
    text = text.lower()
    return re.sub(r'[^a-zmwáàãâéêíóôõúüçñ]+', ' ', text)

# Vetorização e SVD
# max_df=0.5: ignora palavras que aparecem em mais de 50% dos textos (muito comuns)
tfidf = TfidfVectorizer(preprocessor=simple_clean, max_df=0.5, min_df=5, stop_words=stopwords.words('portuguese')) 
X = tfidf.fit_transform(texts)

# Tentamos extrair 6 tópicos para ver se batem com as 6 categorias
svd = TruncatedSVD(n_components=6, random_state=42)
svd.fit(X)

# Exibir as palavras principais de cada componente
terms = tfidf.get_feature_names_out()

print("\n--- TÓPICOS ENCONTRADOS PELO SVD ---")
for i, comp in enumerate(svd.components_):
    terms_comp = zip(terms, comp)
    # Ordena pelos termos com maior peso positivo neste componente
    sorted_terms = sorted(terms_comp, key=lambda x: x[1], reverse=True)[:8]
    print(f"Tópico {i+1}: {', '.join([t[0] for t in sorted_terms])}")

7200 documentos carregados.

--- TÓPICOS ENCONTRADOS PELO SVD ---
Tópico 1: lula, presidente, ex, temer, federal, dilma, lava, jato
Tópico 2: norte, coreia, eua, trump, kim, nuclear, sul, un
Tópico 3: lula, norte, moro, coreia, ex, juiz, eua, kim
Tópico 4: norte, coreia, stf, temer, ministro, lava, jato, supremo
Tópico 5: dilma, lula, temer, presidente, impeachment, rousseff, michel, pt
Tópico 6: odebrecht, milhões, delação, dilma, lava, jato, marcelo, operação


### Análise dos Tópicos Inferidos pelo SVD

Ao analisarmos os componentes gerados, percebe-se que a categoria **Política**, por ser tão extensa (58% do dataset), acabou "engolindo" as demais categorias (como TV, Ciência e Religião). O SVD, buscando maximizar a explicação da variância, dedicou seus vetores para encontrar diferentes nuances (sub-tópicos) dentro dessa categoria dominante:

* **Tópico 1 (Política Geral):** Representa o eixo principal de maior variância. Como a maioria das notícias é sobre política, o vocabulário político domina este vetor.
* **Tópico 2 (Tensão Nuclear):** Fortemente associado a termos como "Coreia do Norte", "EUA" e "Nuclear". Dado que o dataset compreende o período de 2016 a 2018 (ápice da tensão entre Trump e Kim Jong-un), o SVD identificou este assunto como um padrão de vocabulário muito distinto (ortogonal) da política interna brasileira.
* **Tópicos 3 e 4 (Ruído/Mistura):** Aparentam ser uma mistura de vetores. O SVD teve dificuldades em encontrar eixos perfeitamente temáticos e acabou misturando notícias de política nacional e internacional para satisfazer a restrição matemática de ortogonalidade.
* **Tópico 5 (Impeachment):** Focado no processo de impeachment da ex-presidente Dilma Rousseff, um evento de enorme relevância e repetição de vocabulário específico no período.
* **Tópico 6 (Lava-Jato):** Focado em termos como "operação", "delação" e "Odebrecht", separando o vocabulário policial/jurídico do vocabulário legislativo.

**Conclusão:** A restrição de apenas 6 componentes (`n_components=6`) faz com que o SVD fique "preso" aos sub-tópicos da categoria maior. A proporção desbalanceada das categorias impede que o algoritmo reconheça tópicos menores (como Ciência com 1.5%) nas primeiras dimensões. Portanto, para reproduzir a categorização humana proposta no artigo original, o SVD aplicado dessa forma direta não seria a ferramenta adequada.